# Cvičenie 2: Návrh a implementácia vlastných prostredí

Prostredia sú neoddeliteľnou súčasťou problémov riešených *reinforcement learning*om. Kým knižnica `gymnasium` obsahuje mnoho štandardných a často používaných testovacích prostredí, tie nám pri riešení reálnych problémov nepomôžu. Práve preto sa na dnešnom cvičení pozrieme na to, ako navrhnúť a implementovať vlastné prostredia s rovnakým rozhraním ako majú prostredia v `gymnasium`.

## River Crossing

Akí príklad uvažujme klasický problém *River Crossing* (prechod cez rieku s vlkom, kozou a kapustou), ktorý je možné formulovať ako diskrétne prostredie pre reinforcement learning. Prostredie pozostáva z dvoch brehov rieky a malej loďky, pomocou ktorej farmár preváža jednotlivé objekty medzi brehmi. Úlohou agenta je nájsť správnu postupnosť krokov, ktorá bezpečne presunie všetky objekty na opačný breh.

V prostredí vystupujú štyri entity: farmár, vlk, koza a kapusta. Loďka dokáže naraz previezť farmára a najviac jeden ďalší objekt. Problém je obmedzený pravidlami: vlk nesmie zostať s kozou bez prítomnosti farmára a rovnako koza nesmie zostať s kapustou bez farmára. Ak sa takáto situácia vyskytne, stav je považovaný za neplatný.

Stav prostredia je možné reprezentovať ako kombináciu pozícií jednotlivých entít, pričom každá z nich sa nachádza buď na ľavom, alebo pravom brehu rieky. Agent v každom kroku volí akciu — rozhoduje sa, či farmár prejde sám alebo vezme so sebou jeden z objektov. Cieľom je dosiahnuť stav, v ktorom sa všetky entity nachádzajú na cieľovom brehu, pričom počet krokov by mal byť minimálny.

Na prvý pohľad ide o jednoduchú logickú hádanku, avšak z pohľadu reinforcement learningu predstavuje zaujímavý problém plánovania v diskrétnom stavovom priestore s obmed stavom.

## Krok 1: Definícia prostredia

Pred tým, než sa spustíme do implementácie prostredia, potrebujeme ho zadefinovať. Prostredia v kontexte *reinforcement learning*u vieme definovať ako Markovovské rozhodovacie procesy, teda formálne ako štvoricu hodnôt $< S, A, R, P >$ (posledný člen sa často označuje aj ako $T$).

$S$ je stavový priestor prostredia, teda množina všetkých možných stavov, v ktorom sa prostredie môže nachádzať. S definíciou stavového priestoru súvisí aj spôsob reprezentácie stavu, teda pozorovanie - informácie, ktoré budú dostupné agentovi pri rozhodovaní.

Množina akcií $A$ definuje akcie dostupné agentovi. Môže byť definovaná dvomi základnými spôsobmi:

1. je dostupná každá akcia v každom stave prostredia a agent musí zistiť, ktoré akcie spôsobujú zmeny v rôznych stavoch;
2. pre každý stav sa definuje množina platných akcií, v tomto prípade sa agent nemusí naučiť platné akcie.

Funkcia odmeny $R$, ktorá určuje odmenu, ktorú agent dostane za vykonanie istej akcie v istom stave prostredia. Hodnota funkcie môže vychádzať iba zo stavu, alebo z dvojice stav-akcia.

Pre úplnú definíciu prostredia je potrebné určiť aj stavové prechody formou prechodovej matice $P$. Môžeme ju vnímať aj ako mapovaciu funkciu, ktorá na základe súčasného stavu prostredia (a vykonanej akcie) pre každý možný stav prostredia určí pravdepodobnosť toho, že nasledujúci stav prostredia bude práve daný stav.

**Úloha:** Zadefinujte nasledujúce charakteristiky vašej implementácie prostredia River Crossing:

* stavový priestor - čo bude obsahovať táto množina, a ako budú reprezentované jednotlivé stavy? Bude prostredie diskrétne alebo spojité?
* priestor akcií - aké akcie sú dostupné agentovi? Vedeli by sme navrhnúť aj spojité akcie.
* funkcia odmeny - spôsob odmeňovania: agent má vyriešiť problém najmenším možným počtom krokov;
* prechodová matica - pre jednoduchosť bude prostredie úplne deterministické, nový stav prostredia určíme na základe aktuálneho stavu a akcie.

## Krok 2: Definícia triedy

Po definovaní charakteristík prostredia môžeme pristúpiť k implementácii úlohy *River Crossing*. Vytvorte nový Python skript, alebo použite [pripravenú kostru implementácie prostredia](lab02/river_crossing.py). 
Prostredie implementujeme ako triedu reprezentujúcu diskrétne RL prostredi

Na začiatku implementujeme konštruktor, ktorý inicializuje základné vlastnosti problému. Upravte takisto hodnoty premenných tried `action_space` a `observation_space`. Okrem toho v konštruktore vytvorte členské premenné pre vyjadrenie aktuálneho stavu prostredia, a či epizóda má byť dokončená. Naznačte aj cieľový stav prostredia.te náhodne.

In [ ]:
class RiverCrossingEnv():

    action_space = None
    observation_space = None


    def __init__(self):
        pass


    def reset(self, *, seed = None, options = None):
        pass
        # return observation, info


    def step(self, action):
        pass
        # return observation, reward, terminated, truncated, info


    def render(self):
        pass

## Krok 3: Inicializácia prostredia

Po inicializácii prostredia implementujeme funkciu `reset`, ktorá slúži na uvedenie prostredia do počiatočného stavu pred začiatkom novej epizódy učenia. Táto funkcia je volaná vždy na začiatku tréningu alebo po ukončení predchádzajúcej epizódy.

Metóda `reset` nastaví aktuálny stav prostredia na počiatočný ste`. Zároveň je potrebné obnoviť interné premenné prostredia, najmä príznak indikujúci ukončenie epizódy, aby agent začínal vždy z konzistentnej konfigurácie.

Funkcia vracia akte pozorovanieáciu prostrtavu. Spolpozorovanímciou sa vracia aj prázdny informačný slovník, čím je zachovaná kompatibilita s rozhra`ním Gymna`sium.

Implementácia funkcie `reset` teda zabezpečuje, že každá epizóda začína rovnakou, platnou konfiguráciou problému, v ktorej sa všetky entity nachádzajú na počiatočnom brehu rieky a agent môže začať hľadať sekvenciu akcií vedúcich k cieľovému stavu.

## Krok 4: Určenie stavu epizódy

Súčasťou implementácie prostredia je aj funkcia pre určenie stavu epizódy, ktorá overuje, či daný stav predstavuje bezpečnú konfiguráciu problému. Táto funkcia implementuje základné pravidlá úlohy River Crossing a určuje, či v aktuálnom stave nedošlo k porušeniu obmedzení medzi jednotlivými entitami.

Funkcia prijíma stav prostredia zakódovaný ako celé číslo a najskôr ho dekóduje na pozície jednotlivých entít — farmára, vlka, kozy a kapusty. Každá entita sa môže nachádzať na jednom z dvoch brehov rieky. Následne sa kontrolujú nebezpečné situácie, ktoré môžu nastať v neprítomnosti farmára.

Neplatný stav nastáva v prípade, ak sa vlk nachádza na rovnakom brehu ako koza bez prítomnosti farmára, pretože v takom prípade by vlk kozu zožral. Rovnako je stav neplatný, ak sa koza nachádza spolu s kapustou bez farmára. Ak sa nevyskytne ani jedna z týchto situácií, stav je považovaný za platný.

Futate` sa používa pri vyhodnocovaní prechodov medzi stavmi aj pri výpočte odmeny a umožňuje prostrediu identifikovať nebezpečné konfigurácie, ktoré vedú k penalizácii alebo ukončeniu epizódy.

## Krok 5: Výpočet odmeny

Ďalším krokom implementácie je definovanie funkcie odmeny, ktorá určuje, akú spätnú väzbu agent dostane po vykonaní akcie. Funkcia odmeny predstavuje kľúčovú časť prostredia, pretože určuje, aké správanie bude agent počas učenia preferovať.

Funkcia odmeny vyhodnocuje prechod zo starého stavu do nového stavu prostredia. Pri návrhu odmeňovania je potrebné zohľadniť niekoľko situácií. Ak vykonaná akcia nespôsobí zmenu stavu (napríklad ide o neplatnú akciu), agent dostane negatívnu odmenu, ktorá penalizuje neefektívne rozhodnutia. Ak agent dosiahne cieľový stav, teda všetky entity sa bezpečne nachádzajú na cieľovom brehu, prostredie pridelí kladnú odmenu reprezentujúcu úspešné vyriešenie úlohy.

V prípade, že nový stav porušuje pravidlá problému (napríklad koza zostane bez farmára s vlkom alebo kapustou), agent dostane výraznú negatívnu odmenu. Takýto stav reprezentuje zlyhanie a má agenta odradiť od nebezpečných konfigurácií. Vo všetkých ostatných prípadoch je udelená malá záporná odmena za vykonaný krok, ktorá motivuje agenta nájsť riešenie s čo najmenším počtom presutreby.

**Poznámka:** Niekedy funkcia odmeny sa implementuje mimo prostredia, čo umožňuje prácu s rôznymi funkciami odmeny.

## Krok 6: Aktualizácia stavu prostredia

Poslednou kľúčovou časťou implementácie prostredia je funkcia `step`, ktorá reprezentuje jeden interakčný krok medzi agentom a prostredím. Úlohou tejto funkcie je spracovať akciu zvolenú agentom, aktualizovať stav prostredia a vrátiť spätnú väzbu potrebnú pre učenie.

Funkcia `step` prijíma ako vstup akciu, ktorá určuje, koho farmár prevezie na opačný breh rieky (prípadne či sa presunie sám). Najskôr je potrebné uložiť aktuálny stav prostredia a overiť, či je zvolená akcia platná vzhľadom na aktuálne rozmiestnenie entít. Ak akcia nie je platná, stav prostredia sa nezmení.

V prípade platnej akcie sa vypočíta nový stav prostredia vykonaním príslušného prechodu — farmár vždy mení breh a v prípade potreby spolu s ním prejde aj vybraný objekt. Po aktualizácii stavu sa pomocou funkcie odmeny vyhodnotí prechod zo starého stavu do nového stavu.

Následne funkcia určí, či došlo k ukončeniu epizódy. Epiz zatiaľóda sa iba končí dosiahnutím cieľového  ácie).

Funkciap` teda má návratové hodnoty:

1. aktualizovaný stav prostredia
2. odmena za práve vykonaný krok
3. stav interakcie (úspešne ukončená/neukončená)
4. ukončenie interakcie (napr. pri dosiahnutí maximálneho počtu krokov)
5. dodatočné informácie

## Krok 7: Výpis

Na záver ešte môžete vytvoriť funkciu `render`, ktorá vykreslí na obrazovku jednoduchú vizualizáciu stavu prostredia ako aj informácie o poslednom kroku. Grafický výstup môže byť jednoduchý výpis do konzoly, ktorý neskôr môžete upraviť podľa vlastných predstáv.

## Testovanie

Vaše riešenie otestujte vytvorením príkladu vykonaním niekoľkých akcií. Skontrolujte pritom správnosť aktualizácie stavov, ukončenie interakcie a odmeňovanie.

In [ ]:
if __name__ == '__main__':
    env = RiverCrossingEnv()
    print(env)

## Rozšírenie

Vašu implementáciu rozšírte nasledovným spôsobom:

1. Pridajte ukončenie epizódy po dosiahnutí maximálneho počtu krokov.
2. Upravte prechodovú funkciu tak, aby sa epizóda ukončila už pri dosiahnutí neplatného stavu.
3. Navrhnite nedeterministickú prechodovú funkciu (napr. vlk nie je až taký hladný, nie vždy zožerie kozu).